# Titanic Survival Prediction – Clean Notebook

In [ ]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, classification_report

In [ ]:
train_path = "train.csv"
test_path = "test.csv"

train_df = pd.read_csv(train_path)
test_df = pd.read_csv(test_path)

train_df.head()

In [ ]:
cols_to_drop = ["PassengerId", "Ticket", "Cabin", "Name"]
train_df = train_df.drop(columns=cols_to_drop)

test_passenger_id = test_df["PassengerId"]
test_df = test_df.drop(columns=cols_to_drop)

X = train_df.drop("Survived", axis=1)
y = train_df["Survived"]

numeric_features = ["Age", "SibSp", "Parch", "Fare", "Pclass"]
categorical_features = ["Sex", "Embarked"]

In [ ]:
numeric_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="median"))
    ]
)

categorical_transformer = Pipeline(
    steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ]
)

preprocessor = ColumnTransformer(
    transformers=[
        ("num", numeric_transformer, numeric_features),
        ("cat", categorical_transformer, categorical_features),
    ]
)

model = RandomForestClassifier(
    n_estimators=200,
    random_state=42,
    max_depth=None,
    n_jobs=-1
)

clf = Pipeline(
    steps=[
        ("preprocessor", preprocessor),
        ("model", model)
    ]
)

In [ ]:
X_train, X_valid, y_train, y_valid = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

clf.fit(X_train, y_train)

y_pred = clf.predict(X_valid)
print("Validation accuracy:", accuracy_score(y_valid, y_pred))
print(classification_report(y_valid, y_pred))

In [ ]:
clf.fit(X, y)

test_pred = clf.predict(test_df)

submission = pd.DataFrame({
    "PassengerId": test_passenger_id,
    "Survived": test_pred
})

submission.head()

In [ ]:
submission.to_csv("submission.csv", index=False)
"Saved submission.csv"